In [1]:
import duckdb
from ga4case.path import EVENTS_FLATTENED

In [2]:
con = duckdb.connect()
FLAT = EVENTS_FLATTENED.as_posix()
con.sql(f"create or replace view events as select * from read_parquet('{FLAT}')")

CountBIGINT


In [3]:
# Check whether each event recorded across the three months: events per visit by week
weekly = con.sql("""
    with visits as (
        select date_trunc('week',event_date)::date as week,
            count (distinct event_date) as days,
            count (distinct (user_pseudo_id, ga_session_id)) as visits
        from events
        group by week
        ),
    counts as (
        select date_trunc('week', event_date):: date as week,
            event_name,
            count(*) n_events
        from events
        group by week, event_name
        )
        select counts.week, days, visits, event_name,
            round(n_events/visits, 4) as events_per_visit
        from counts join visits using (week)
        order by counts.week, event_name
    """).df()

In [4]:
weekly.pivot(
    index=["week", "days", "visits"], columns="event_name", values="events_per_visit"
).fillna(0)

,,event_name,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,page_view,purchase,scroll,select_item,select_promotion,session_start,user_engagement,view_item,view_item_list,view_promotion,view_search_results
week,days,visits,,,,,,,,,,,,,,,,,
2020-10-26,1,2625,0.0278,0.0484,0.0015,0.0484,0.0042,0.7413,3.7208,0.0053,1.1581,0.0000,0.0217,0.9882,3.3017,1.2876,0.0030,0.4811,0.0739
2020-11-02,7,24996,0.0352,0.0566,0.0000,0.0566,0.0077,0.6605,4.0535,0.0116,1.4685,0.0000,0.0292,0.9874,3.6461,1.3415,0.0009,0.6719,0.0681
2020-11-09,7,22519,0.0532,0.0738,0.0000,0.1034,0.0099,0.6483,4.3059,0.0162,1.6164,0.0000,0.0262,0.9872,3.9181,1.4166,0.0005,0.6184,0.0896
2020-11-16,7,25923,0.0457,0.0657,0.0557,0.0996,0.0091,0.6525,4.2188,0.0218,1.6054,0.0000,0.0301,0.9891,3.8022,1.3670,0.0002,0.6341,0.0902
2020-11-23,7,27788,0.0512,0.0754,0.1605,0.0771,0.0031,0.6745,4.1545,0.0244,1.6002,0.0668,0.0261,0.9689,3.6646,1.3408,0.0001,0.5636,0.0804
2020-11-30,7,31098,0.0543,0.0817,0.3082,0.1602,0.0033,0.6820,4.2614,0.0207,1.6643,0.1449,0.0292,0.9896,3.7563,1.4139,0.0002,0.5850,0.0800
2020-12-07,7,38739,0.0544,0.0746,0.2810,0.1473,0.0037,0.7117,3.8763,0.0223,1.4522,0.1514,0.0262,0.9844,3.3680,1.2595,0.0001,0.5696,0.0742
2020-12-14,7,33419,0.0495,0.0649,0.2923,0.1486,0.0037,0.7252,3.3805,0.0205,1.3700,0.1515,0.0273,0.9902,2.8705,1.1216,0.0000,0.5145,0.0706
2020-12-21,7,22094,0.0262,0.0367,0.2192,0.1092,0.0034,0.7703,2.8177,0.0116,1.1253,0.1134,0.0242,0.9889,2.3114,0.8554,0.0000,0.4487,0.0638


Findings:
1. add_to_cart might be reliable from the end of November. -- The event might be not recorded.
2. select_item are 0 until Nov 23, went up at Nov 30.
3. page_view rises in late December but user_engagement and item_view falls. Therefore, this change may be artefacts.

In [5]:
# Check whether a user's channel stay the same: test whether the channel tells where a user came from
con.sql("""
    with visits as (
        select user_pseudo_id, ga_session_id,
            min(event_timestamp) as visit_start,
            min(traffic_source || '/' || traffic_medium) as channel
        from events
        group by user_pseudo_id, ga_session_id
        ),
    ordered as (
        select *, row_number() over (partition by user_pseudo_id order by visit_start) as visit_order
        from visits
        )
    select count(*) as users_with_two_visits,
        round(100* avg((first.channel <> second.channel):: int), 1) as pct_channel_differs
    from ordered as first
    join ordered as second
      on first.user_pseudo_id = second.user_pseudo_id
    and first.visit_order = 1
    and second.visit_order =2
    """).df()

,users_with_two_visits,pct_channel_differs
0,47364,83.8


Finding: The channel on first and second visit differes for 83.8% of users, thus the channel cannot be used

In [6]:
# Check the consistency of the device and country: eg. any difference in visitors using desktop, mobile or tablet in different countris
con.sql("""
    with visits as (
        select user_pseudo_id, ga_session_id,
        min(geo_country) as country,
        min(device_category) as device
    from events
    group by user_pseudo_id, ga_session_id
    )
    select country,
        count(*) as visits,
        round (100 * avg((device = 'desktop'):: int), 1) as pct_desktop,
        round (100 * avg((device = 'mobile'):: int), 1) as pct_mobile,
        round (100 * avg((device = 'tablet'):: int), 1) as pct_tablet
    from visits
    group by country
    order by visits desc
    limit 10
    """).df()


,country,visits,pct_desktop,pct_mobile,pct_tablet
0,United States,158155,58.1,39.7,2.2
1,India,33769,58.0,39.7,2.3
2,Canada,26824,57.6,40.3,2.1
3,United Kingdom,11327,58.1,39.3,2.6
4,France,7162,59.5,38.6,2.0
5,Spain,6667,58.8,38.9,2.3
6,Germany,6393,57.5,40.3,2.2
7,China,6258,57.0,40.4,2.5
8,Taiwan,6057,57.4,40.5,2.1
9,Italy,4998,58.1,39.7,2.2


Finding: the device mix is similar across the 10 countries -- may not be true in reality.

In [7]:
# Check whether there is a trend of the day
con.sql("""
    with visits as (
        select user_pseudo_id, ga_session_id,
        min(event_timestamp) as visit_start,
        min(geo_country) as country
    from events
    group by user_pseudo_id, ga_session_id
    )
    select hour(visit_start) as hour_of_day,
        round(100*count(*) /sum(count(*)) over (), 2) as pct_all,
        round(100*count(*) filter (where country = 'India') /sum(count(*) filter (where country = 'India')) over (), 2) as pct_india,
        round(100*count(*) filter (where country = 'Canada') /sum(count(*) filter (where country = 'Canada')) over (), 2) as pct_canada
    from visits
    group by hour_of_day
    order by hour_of_day 
    """).df()

,hour_of_day,pct_all,pct_india,pct_canada
0,0,4.17,4.32,4.14
1,1,4.18,4.26,4.04
2,2,4.16,4.25,4.16
3,3,4.16,4.27,4.20
4,4,4.12,4.10,4.13
5,5,4.24,4.15,4.33
6,6,4.18,4.25,3.91
7,7,4.13,4.10,4.07
8,8,4.16,4.25,4.04
9,9,4.23,4.09,4.20


Finding: Similar visits within every hour in different countries.

In [8]:
con.sql("""
    select dayname(event_date) as weekday, isodow(event_date) as d,
        round(100* count(distinct (user_pseudo_id, ga_session_id)) /sum(count(distinct (user_pseudo_id, ga_session_id))) over (), 1) as pct_visits
    from events group by 1,2 order by d 
    """).df()

,weekday,d,pct_visits
0,Monday,1,14.4
1,Tuesday,2,16.9
2,Wednesday,3,16.7
3,Thursday,4,15.5
4,Friday,5,14.1
5,Saturday,6,11.2
6,Sunday,7,11.2


Finding: Visits of 16.9% on Tuesday dropped to 11.2% on weekends.

### Summary from checks:
**Use**
- Date and weekday: weekly pattern.

**Use cautiously**
- Cart events: only from later November could be used.
- Page_view: late December should be careful treated.

**Not use**
- channel:  differs for 83.8% users between visit.
- daily hours: all around 4.2% every hour.
- device and countries: mixe similarly among 10 countries.
